[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C72_MultiView_Geometry_Course/04_bev_fusion/04_bev_fusion.ipynb)

# 04 · BEV 投影与多相机融合

四件事：

1. **把 IPM 建成一个反向映射**，并验证地面点闭环。
2. **量出均匀网格的采样率崩塌**：3.33–10 m 过采样 5.4×，
   50–100 m 欠采样 **27.8×**；以及前向映射在那一档留下 **96.4% 的空洞**。
3. **坡度：1% 在 50 m 处值 +50%**，临界距离 $H/s$ —— 本模块的中心结论。
4. **重叠区一致性**：侧相机 pitch 错 0.5°，50 m 处两路差 **22.6%** ——
   这是这一层唯一不需要真值、且行驶中每帧都有的指标。

## 0 · 环境

In [ ]:
import numpy as np

print('numpy', np.__version__)

H_CAM = 1.5
F, CX, CY = 1200.0, 960.0, 540.0
W, HGT = 1920, 1080

def R_vc(pitch_deg=0.0):
    t = np.deg2rad(pitch_deg)
    return np.array([[0., -np.sin(t),  np.cos(t)],
                     [-1.,        0.,        0.],
                     [0., -np.cos(t), -np.sin(t)]])

CAM_T = np.array([0., 0., H_CAM])

def project(P_v, pitch=0.0, cam_t=None):
    cam_t = CAM_T if cam_t is None else np.asarray(cam_t, float)
    P_c = R_vc(pitch).T @ (np.asarray(P_v, float) - cam_t)
    if P_c[2] <= 1e-9:
        return None
    return np.array([CX + F * P_c[0] / P_c[2], CY + F * P_c[1] / P_c[2]])

def ground_from_pixel(uv, pitch=0.0, cam_t=None, plane_z=0.0):
    '''反投影到 z=plane_z 的平面。返回 None 表示无交点。'''
    cam_t = CAM_T if cam_t is None else np.asarray(cam_t, float)
    d_c = np.array([(uv[0] - CX) / F, (uv[1] - CY) / F, 1.0])
    d_v = R_vc(pitch) @ d_c
    if abs(d_v[2]) < 1e-12:
        return None
    s = (plane_z - cam_t[2]) / d_v[2]
    if s <= 0:
        return None
    return cam_t + s * d_v

d_near = H_CAM * F / (HGT - CY)
print(f'画幅内能看到的地面从 {d_near:.2f} m 开始（v={HGT}）')
print(f'地平线在 v={CY:.0f}，对应无穷远')

## 1 · IPM 是一个单应，而且必须反向映射

BEV 网格的每一格 → 车体坐标 → 图像坐标 → 采样。
**方向是「从 BEV 去图像取」，不是「从图像往 BEV 填」。**

In [ ]:
class BEVGrid:
    '''BEV 的「内参」：每格多少米、覆盖范围。'''
    def __init__(self, x_range=(0., 100.), y_range=(-10., 10.), res=0.1):
        self.x0, self.x1 = x_range
        self.y0, self.y1 = y_range
        self.res = res
        self.nx = int(round((self.x1 - self.x0) / res))
        self.ny = int(round((self.y1 - self.y0) / res))

    def cell_center(self, ix, iy):
        '''格索引 -> 车体坐标 (X前, Y左, 0)。'''
        return np.array([self.x0 + (ix + 0.5) * self.res,
                         self.y0 + (iy + 0.5) * self.res, 0.0])

    def __repr__(self):
        return (f'BEVGrid({self.x0}-{self.x1}m x {self.y0}-{self.y1}m, '
                f'{self.res}m/格, {self.nx}x{self.ny})')

GRID = BEVGrid()
print(GRID)
print(f'共 {GRID.nx * GRID.ny:,} 格')

def inverse_map(grid, pitch=0.0, cam_t=None):
    '''反向映射：为每个 BEV 格算出源图像坐标。返回 (uv, valid)。'''
    ix, iy = np.meshgrid(np.arange(grid.nx), np.arange(grid.ny), indexing='ij')
    X = grid.x0 + (ix + 0.5) * grid.res
    Y = grid.y0 + (iy + 0.5) * grid.res
    P = np.stack([X, Y, np.zeros_like(X)], axis=-1).reshape(-1, 3)
    cam_t = CAM_T if cam_t is None else np.asarray(cam_t, float)
    Pc = (R_vc(pitch).T @ (P - cam_t).T).T
    uv = np.full((len(P), 2), np.nan)
    ok = Pc[:, 2] > 1e-9
    uv[ok, 0] = CX + F * Pc[ok, 0] / Pc[ok, 2]
    uv[ok, 1] = CY + F * Pc[ok, 1] / Pc[ok, 2]
    inside = ok & (uv[:, 0] >= 0) & (uv[:, 0] < W) & (uv[:, 1] >= 0) & (uv[:, 1] < HGT)
    return uv.reshape(grid.nx, grid.ny, 2), inside.reshape(grid.nx, grid.ny)

UVMAP, VALID = inverse_map(GRID)
print(f'\n反向映射：{VALID.sum():,} / {VALID.size:,} 格有对应的图像像素 '
      f'({100*VALID.mean():.1f}%)')

# 闭环：BEV 格中心 -> 图像 -> 反投影回地面，必须回到原处
for ix, iy in [(50, 100), (200, 100), (500, 50), (900, 150)]:
    Pc = GRID.cell_center(ix, iy)
    uv = project(Pc)
    back = ground_from_pixel(uv)
    assert np.allclose(back, Pc, atol=1e-9), (ix, iy, back, Pc)
print('✅ BEV 格 -> 图像 -> 地面 精确闭环（< 1e-9 m）')

# 地平线以上的格子必须被标成无观测，而不是采样到边缘
far = GRID.cell_center(GRID.nx - 1, GRID.ny // 2)
print(f'最远一格 X={far[0]:.1f}m -> v={project(far)[1]:.2f}'
      f'（地平线 {CY:.0f} 之下 {project(far)[1]-CY:.2f} px）')

## 2 · 采样率：均匀网格必然一头过采样、一头欠采样

In [ ]:
def v_of(d):
    return CY + F * H_CAM / d

print(f"{'距离':>7s} {'像素行 v':>10s} {'±1px 覆盖的米数':>17s}")
for d in [d_near, 10., 20., 30., 50., 80.]:
    v = v_of(d)
    dd = abs(H_CAM * F / (v + 1 - CY) - d)
    print(f'{d:6.2f}m {v:10.1f} {dd:16.3f}m')

print(f'\n{"距离段":>16s} {"图像行数":>10s} {"0.1m 格数":>10s} '
      f'{"每格几行":>10s} {"倍数":>10s}')
bands = [(d_near, 10.), (10., 20.), (20., 50.), (50., 100.)]
stats = {}
for lo, hi in bands:
    rows = v_of(lo) - v_of(hi)
    cells = (hi - lo) / GRID.res
    per = rows / cells
    tag = f'过采样 {per:.1f}x' if per > 1 else f'**欠采样 {1/per:.1f}x**'
    stats[(lo, hi)] = (rows, cells, per)
    print(f'{lo:7.2f}–{hi:5.0f}m {rows:9.1f} {cells:10.0f} {per:10.3f} {tag:>14s}')

near = stats[bands[0]][2]
far_ = stats[bands[-1]][2]
assert near > 5.0, f'近处应过采样 5x 以上，实测 {near:.2f}'
assert far_ < 0.05, f'远处应欠采样 20x 以上，实测 1/{1/far_:.1f}'
print(f'\n✅ 近处过采样 {near:.1f}x，远处欠采样 {1/far_:.1f}x '
      f'—— 相差 {near/far_:.0f} 倍')
print('   → **均匀 BEV 网格是一个内在矛盾的设计**')

# 前向映射的空洞率 = 同一件事的另一种说法
print(f'\n前向映射（图像 -> BEV）的空洞率：')
holes = {}
for lo, hi in bands:
    rows, cells, _ = stats[(lo, hi)]
    filled = min(rows, cells)
    holes[(lo, hi)] = 1 - filled / cells
    print(f'  {lo:6.2f}–{hi:5.0f}m: 最多能填 {filled:5.0f}/{cells:.0f} 格'
          f'  -> 空洞率 {100*(1-filled/cells):5.1f}%')
assert holes[bands[-1]] > 0.9, '50–100m 的空洞率应超过 90%'
print(f'\n✅ 50–100 m 有 {100*holes[bands[-1]]:.1f}% 的格子填不上'
      ' —— 所以 IPM 必须反向映射')

# 横向：一格对应多少列
print(f'\n横向：0.1m 宽的格在各距离对应多少图像列')
for d in [5., 10., 20., 50., 80.]:
    print(f'  {d:5.0f}m: {F*GRID.res/d:6.2f} px')

## 3 · 地面假设破坏的代价（回收模块 00 的中心公式）

In [ ]:
def ipm_read(d, z, h=H_CAM):
    '''把离地 z 的目标当地面点处理时读出的距离。'''
    return np.inf if z >= h else d * h / (h - z)

CASES = [
    ('路面起伏 / 减速带',  0.05),
    ('路肩、排水沟',      -0.15),
    ('1% 上坡 @50m',       0.50),
    ('低矮施工牌',         1.00),
    ('限速牌',             2.20),
    ('龙门架标志',         5.50),
]
print(f"{'情形':>18s} {'等效 z':>8s} {'50m 处读出':>12s} {'相对误差':>10s}")
for name, z in CASES:
    r = ipm_read(50., z)
    rel = 'inf' if not np.isfinite(r) else f'{100*(r-50)/50:+.0f}%'
    rs = '无解' if not np.isfinite(r) else f'{r:.1f} m'
    print(f'{name:>18s} {z:7.2f}m {rs:>12s} {rel:>10s}')

# 分界线在 z ≈ 0.15 m：低于它可忽略
assert abs(ipm_read(50., 0.05) - 50.) / 50. < 0.05, 'z=0.05 应当可忽略'
assert ipm_read(50., 0.5) / 50. > 1.4, 'z=0.5 应当致命'
assert ipm_read(50., 2.2) == np.inf, 'z>H 必须无解'
print('\n✅ 分界线在 z≈0.15m：低于它可忽略，高于它很快致命（z/(H−z) 在 z→H 时发散）')

# 检测框底边被当接地点：对车辆近似成立，对标志完全不成立
print('\n同一个函数被用在两类目标上：')
for name, z in [('车辆（轮胎接地）', 0.0), ('交通标志（牌面下沿 1.8m）', 1.8)]:
    r = ipm_read(50., z)
    print(f'  {name:24s} -> {"无解" if not np.isfinite(r) else f"{r:.1f} m"}')
print('  → **IPM 函数必须拿到类别并拒绝「不贴地」的类别**')

## 4 · 中心结论：1% 的坡度值 50%

坡度让地面高度变成 $z=s\cdot d$，代入中心公式得
$d_{\text{read}}=dH/(H-sd)$，临界距离 $H/s$。

In [ ]:
def ipm_read_slope(d, s, h=H_CAM):
    z = s * d
    return np.inf if z >= h else d * h / (h - z)

SLOPES = [0.005, 0.01, 0.02, 0.03, -0.01, -0.02]
DISTS = [20., 30., 50., 80.]
print(f"{'坡度':>7s} {'临界距离':>9s} " + ''.join(f'{d:.0f}m'.rjust(11) for d in DISTS))
for s in SLOPES:
    crit = H_CAM / s if s > 0 else np.inf
    cs = 'inf' if not np.isfinite(crit) else f'{crit:.0f} m'
    row = [ipm_read_slope(d, s) for d in DISTS]
    print(f'{s*100:6.1f}% {cs:>9s} ' +
          ''.join(('       无解' if not np.isfinite(v) else f'{v:10.1f}') for v in row))

# ① 1% 坡度在 50m 处 +50%
r1 = ipm_read_slope(50., 0.01)
print(f'\n1% 上坡 @50m: {r1:.1f} m（真 50 m）→ {100*(r1-50)/50:+.0f}%')
assert abs(r1 - 75.0) < 1e-9, '1% 坡度在 50m 处应精确给出 75 m'

# ② 临界距离 = H/s
for s in [0.01, 0.02, 0.03]:
    crit = H_CAM / s
    assert ipm_read_slope(crit * 0.999, s) < np.inf
    assert ipm_read_slope(crit, s) == np.inf
    print(f'  s={s*100:.0f}%: 临界距离 {crit:.0f} m —— 之外无解')

# ③ 上坡高估远大于下坡低估（分母的不对称）
up, dn = ipm_read_slope(50., 0.01), ipm_read_slope(50., -0.01)
print(f'\n±1% 在 50m 处：上坡 {up:.1f} m（{100*(up-50)/50:+.0f}%）'
      f' vs 下坡 {dn:.1f} m（{100*(dn-50)/50:+.0f}%）')
assert (up - 50) > 1.9 * (50 - dn), '上坡的误差幅度应远大于下坡'
print('  → **上坡高估（不安全方向），下坡低估（安全但误刹）**')

# ④ 应对②：把 IPM 限制在 d < H/(3s)
s_worst = 0.02
d_limit = H_CAM / (3 * s_worst)
err_at_limit = ipm_read_slope(d_limit, s_worst) / d_limit - 1
print(f'\n取最坏坡度 {s_worst*100:.0f}%：限制 IPM 在 {d_limit:.0f} m 以内，'
      f'误差 <= {100*err_at_limit:.0f}%')
assert err_at_limit < 0.55
print('✅ 坡度不是小扰动：1% 值 50%，而 2% 的临界距离只有 75 m')

## 5 · 多相机与重叠区一致性（无真值的外参监控）

In [ ]:
# 主相机在车体中线，侧相机左移 1.0 m（同高度）
CAM_MAIN = np.array([0., 0.,  H_CAM])
CAM_SIDE = np.array([0., 1.0, H_CAM])

def read_range(P_world, cam_t, pitch=0.0):
    '''某一路相机对一个地面点的 IPM 读数（纵向距离）。'''
    uv = project(P_world, pitch=0.0, cam_t=cam_t)      # 真实成像（外参正确）
    if uv is None:
        return None
    g = ground_from_pixel(uv, pitch=pitch, cam_t=cam_t)  # 用（可能错的）pitch 反投影
    return None if g is None else float(g[0])

print('外参正确时，两路读数应当精确相等：')
for d in [10., 20., 30., 50.]:
    Pw = np.array([d, 0.5, 0.])
    a = read_range(Pw, CAM_MAIN, 0.0)
    b = read_range(Pw, CAM_SIDE, 0.0)
    assert abs(a - d) < 1e-9 and abs(b - d) < 1e-9, (d, a, b)
    print(f'  {d:5.0f}m: 主 {a:.6f}  侧 {b:.6f}')
print('✅ 外参正确 -> 两路一致到 1e-9 m（而这不需要任何真值）\n')

PITCH_ERR = 0.5
print(f'把侧相机的 pitch 拧错 {PITCH_ERR}°：')
print(f"{'距离':>6s} {'主相机':>9s} {'侧相机':>9s} {'差值':>9s} {'相对差':>8s}")
rels = []
for d in [10., 20., 30., 50.]:
    Pw = np.array([d, 0.5, 0.])
    a = read_range(Pw, CAM_MAIN, 0.0)
    b = read_range(Pw, CAM_SIDE, PITCH_ERR)
    rel = abs(a - b) / a
    rels.append(rel)
    print(f'{d:5.0f}m {a:8.2f}m {b:8.2f}m {abs(a-b):8.2f}m {100*rel:7.1f}%')

assert rels == sorted(rels), '灵敏度应随距离上升'
assert rels[-1] > 0.20, f'50m 处相对差应超过 20%，实测 {100*rels[-1]:.1f}%'
print(f'\n✅ 0.5° 外参误差 → 50 m 处 {100*rels[-1]:.1f}% 的两路不一致')
print('   → 灵敏度随距离上升，所以监控应在重叠区的**远端**取样')
print(f'   → 而 0.5° 是模块 02 验收阈值(0.05°)的 {PITCH_ERR/0.05:.0f} 倍')

# 它抓不住「共同」误差
print('\n两路一起偏（车体 Z=0 定义错了 5cm）：')
WRONG_H = H_CAM - 0.05
for d in [20., 50.]:
    Pw = np.array([d, 0.5, 0.])
    uv_a = project(Pw, cam_t=CAM_MAIN); uv_b = project(Pw, cam_t=CAM_SIDE)
    a = ground_from_pixel(uv_a, cam_t=np.array([0., 0., WRONG_H]))[0]
    b = ground_from_pixel(uv_b, cam_t=np.array([0., 1., WRONG_H]))[0]
    print(f'  {d:5.0f}m: 主 {a:.3f}  侧 {b:.3f}  差 {abs(a-b):.2e}m'
          f'  （都错了 {100*(a-d)/d:+.1f}%，而一致性检查全过）')
    assert abs(a - b) < 1e-9, '共同误差不会造成不一致'
print('  → **一致性检查只抓「相对」错误，抓不住「共同」错误**')

## 6 · 反向校验：把 BEV 结果投回图像

它抓「代码/外参错」，**抓不住「假设错」**——因为投影与反投影用同一个假设。

In [ ]:
def bev_to_image(P_bev, cam_t=None, pitch=0.0):
    return project(np.asarray(P_bev, float), pitch=pitch, cam_t=cam_t)

print('情形 A：链路自洽（同一套 K/外参）')
for d in [20., 50.]:
    uv0 = project(np.array([d, 0.5, 0.]))
    g = ground_from_pixel(uv0)
    uv1 = bev_to_image(g)
    print(f'  {d:5.0f}m: 原检测 v={uv0[1]:.3f} -> 投回 v={uv1[1]:.3f}'
          f'  差 {abs(uv1[1]-uv0[1]):.2e} px')
    assert abs(uv1[1] - uv0[1]) < 1e-9

print('\n情形 B：反投影用了错的 pitch（代码/配置错）')
for d in [20., 50.]:
    uv0 = project(np.array([d, 0.5, 0.]))
    g = ground_from_pixel(uv0, pitch=0.5)          # 错
    uv1 = bev_to_image(g, pitch=0.0)               # 投回时用对的
    print(f'  {d:5.0f}m: 原 v={uv0[1]:.2f} -> 投回 v={uv1[1]:.2f}'
          f'  **差 {abs(uv1[1]-uv0[1]):.2f} px**')
    assert abs(uv1[1] - uv0[1]) > 1.0, '不一致的外参应当被抓住'
print('  → ✅ 抓住了')

print('\n情形 C：假设错了（把 2.2m 高的标志当地面点）')
Pw = np.array([50., -3.2, 2.2])
uv0 = project(Pw)
g = ground_from_pixel(uv0)
if g is None:
    print(f'  z=2.2m > H：反投影直接无解 —— 这一次假设错误是「响的」')
else:
    uv1 = bev_to_image(g)
    print(f'  原 v={uv0[1]:.2f} -> 投回 v={uv1[1]:.2f}  差 {abs(uv1[1]-uv0[1]):.2e} px')

Pw2 = np.array([50., -3.2, 1.0])                   # 换一个 z<H 的
uv0 = project(Pw2); g = ground_from_pixel(uv0); uv1 = bev_to_image(g)
print(f'  改用 z=1.0m 的低矮牌：原 v={uv0[1]:.2f} -> 投回 v={uv1[1]:.2f}'
      f'  差 {abs(uv1[1]-uv0[1]):.2e} px')
print(f'  而它的 BEV 位置错了 {100*(g[0]-50)/50:+.0f}%（读出 {g[0]:.1f} m）')
assert abs(uv1[1] - uv0[1]) < 1e-9, '假设错误时投影仍然精确闭环'
assert abs(g[0] - 50.) / 50. > 1.9
print('  → ❌ **抓不住**：投影与反投影用同一个错误假设，所以它精确闭环')
print('\n✅ 三类错误、三种手段：闭环抓代码错 · 重叠区抓外参错 · '
      '类别拒绝与坡度处理抓假设错')

## 7 · 小结

| 结论 | 数值 |
|---|---|
| 可见地面范围 | 从 **3.33 m** 开始，到 100 m 只占 522 个图像行 |
| 均匀网格采样率 | 近处过采样 **5.4×**，远处欠采样 **27.8×**（相差 150 倍） |
| 前向映射空洞率 | 50–100 m 有 **96.4%** 的格填不上 → 必须反向映射 |
| 地面假设的分界线 | $z\approx0.15$ m；$z\ge H$ 时**无解** |
| **坡度** | 1% 在 50 m 处 **+50%**；临界距离 $H/s$，2% 时只有 **75 m** |
| 坡度方向 | 上坡高估（不安全），且幅度远大于下坡低估 |
| **重叠区一致性** | 0.5° 外参误差 → 50 m 处 **22.6%**，灵敏度随距离上升 |
| 它的盲区 | **共同误差不产生不一致**（车体 $Z=0$ 定义错了照样全过） |
| 反向校验 | 抓代码/外参错；**抓不住假设错**（精确闭环） |

## ✏️ 练习 1：非均匀 BEV 网格

均匀网格近处过采样 5.4×、远处欠采样 27.8×。
实现 `log_bev_edges(d_near, d_far, n_cells)`：给出 `n_cells+1` 个纵向网格边界，
使得**每格对应的图像行数大致相同**。

提示：图像行 $v=c_y+fH/d$ 与 $1/d$ 成线性，
所以「等图像行」等价于「$1/d$ 等分」。

In [ ]:
def log_bev_edges(d_near, d_far, n_cells):
    """返回长度 n_cells+1 的距离边界数组，使每格占的图像行数相同。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
N = 200
edges = log_bev_edges(d_near, 100., N)
assert len(edges) == N + 1
assert abs(edges[0] - d_near) < 1e-9 and abs(edges[-1] - 100.) < 1e-9
assert np.all(np.diff(edges) > 0), '边界必须单调递增'

rows_per = np.array([v_of(edges[i]) - v_of(edges[i+1]) for i in range(N)])
uni = np.linspace(d_near, 100., N + 1)
rows_uni = np.array([v_of(uni[i]) - v_of(uni[i+1]) for i in range(N)])

print(f"{'网格':>10s} {'每格行数 min':>13s} {'max':>9s} {'max/min':>9s}")
print(f'{"非均匀":>10s} {rows_per.min():12.4f} {rows_per.max():9.4f} '
      f'{rows_per.max()/rows_per.min():9.2f}')
print(f'{"均匀":>10s} {rows_uni.min():12.4f} {rows_uni.max():9.4f} '
      f'{rows_uni.max()/rows_uni.min():9.1f}')

assert rows_per.max() / rows_per.min() < 1.01, \
    f'非均匀网格应当近乎等行数，实测 {rows_per.max()/rows_per.min():.3f}'
assert rows_uni.max() / rows_uni.min() > 100, '均匀网格的比值应当很大'

# 代价：近处格子变粗
print(f'\n非均匀网格的格宽（米）：'
      f'最近 {edges[1]-edges[0]:.3f}m，最远 {edges[-1]-edges[-2]:.3f}m')
print(f'均匀网格：处处 {uni[1]-uni[0]:.3f}m')
assert (edges[1]-edges[0]) < (uni[1]-uni[0]), '非均匀网格近处更细'
assert (edges[-1]-edges[-2]) > (uni[-1]-uni[-2]) * 5, '而远处更粗'
print('✅ 练习 1 通过：等图像行 = 1/d 等分；代价是远处的格子在米域变粗')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def log_bev_edges(d_near, d_far, n_cells):
    # v = CY + F*H/d 与 1/d 线性，所以在 1/d 上等分即得等行数
    inv = np.linspace(1.0 / d_near, 1.0 / d_far, n_cells + 1)
    return 1.0 / inv

e = log_bev_edges(d_near, 100., 200)
r = np.array([v_of(e[i]) - v_of(e[i+1]) for i in range(200)])
assert r.max() / r.min() < 1.01
print(f'每格 {r.mean():.4f} 行（比值 {r.max()/r.min():.4f}）')
print('✅ 参考答案 1 通过（这也解释了为什么 BEV 里常见「远处格子更大」的设计）')

## ✏️ 练习 2：带坡度的 IPM 与作用距离上限

实现两个函数：

- `ipm_with_slope(v_px, slope)` —— 已知坡度时的正确读数（返回 `None` 表示无解）
- `ipm_max_range(slope_worst, max_rel_err)` —— 给定最坏坡度与可接受的相对误差，
  返回 IPM 的作用距离上限

In [ ]:
def ipm_with_slope(v_px, slope, h=H_CAM):
    """射线与斜面 z = slope*x 的交点的纵向距离；无解返回 None。"""
    # TODO
    raise NotImplementedError

def ipm_max_range(slope_worst, max_rel_err, h=H_CAM):
    """未建模坡度为 slope_worst 时，相对误差不超过 max_rel_err 的最大距离。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# ① 坡度已知时，读数应当精确
for d in [20., 50., 80.]:
    for s in [0.0, 0.01, 0.02, -0.01]:
        z = s * d
        uv = project(np.array([d, 0., z]))
        got = ipm_with_slope(uv[1], s)
        assert got is not None and abs(got - d) < 1e-6, (d, s, got)
print('✅ 坡度已知时 ipm_with_slope 精确复原（误差 < 1e-6 m）')

# ② 坡度未建模（按平地读）时的误差，就是中心公式
for s in [0.01, 0.02]:
    for d in [30., 50.]:
        uv = project(np.array([d, 0., s * d]))
        flat = ipm_with_slope(uv[1], 0.0)
        assert abs(flat - ipm_read_slope(d, s)) < 1e-6, (s, d, flat)
print('✅ 按平地读时与 d·H/(H−sd) 一致')

# ③ 作用距离上限
print(f"\n{'最坏坡度':>9s} {'容许相对误差':>13s} {'距离上限':>10s} {'在上限处的实际误差':>19s}")
for s in [0.01, 0.02, 0.03]:
    for tol in [0.2, 0.5]:
        lim = ipm_max_range(s, tol)
        act = ipm_read_slope(lim, s) / lim - 1
        print(f'{s*100:8.0f}% {tol:12.0%} {lim:9.1f}m {act:18.1%}')
        assert abs(act - tol) < 0.02, f'上限处的误差应当接近容许值（{act:.3f} vs {tol}）'

# 上限随坡度反比、随容许误差单调
assert ipm_max_range(0.01, 0.2) > ipm_max_range(0.02, 0.2)
assert ipm_max_range(0.02, 0.5) > ipm_max_range(0.02, 0.2)
lim2 = ipm_max_range(0.02, 0.5)
print(f'\n2% 最坏坡度 + 容许 50% 误差 -> IPM 上限 {lim2:.1f} m'
      f'（而临界距离是 {H_CAM/0.02:.0f} m）')
assert lim2 < H_CAM / 0.02, '上限必须小于临界距离'
print('✅ 练习 2 通过：坡度从「未建模误差」变成「一个参数」或「一个距离上限」')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def ipm_with_slope(v_px, slope, h=H_CAM):
    # 射线（pitch=0）：X = t, Z = h - t*(v-CY)/F
    # 斜面：Z = slope * X
    # 解得 t*(slope + (v-CY)/F) = h
    k = (v_px - CY) / F
    den = slope + k
    if den <= 1e-12:
        return None
    return h / den

def ipm_max_range(slope_worst, max_rel_err, h=H_CAM):
    # 未建模坡度下 d_read/d = h/(h - s*d)，令它等于 1+tol
    # => h = (1+tol)(h - s*d) => d = tol*h / (s*(1+tol))
    return max_rel_err * h / (slope_worst * (1 + max_rel_err))

uv = project(np.array([50., 0., 0.5]))
assert abs(ipm_with_slope(uv[1], 0.01) - 50.) < 1e-6
lim = ipm_max_range(0.02, 0.5)
assert abs(ipm_read_slope(lim, 0.02) / lim - 1 - 0.5) < 1e-9
print(f'2%/50% -> 上限 {lim:.2f} m')
print('✅ 参考答案 2 通过（注意 ipm_with_slope 的分母 slope+k：'
      '上坡让分母变大 -> 距离变小，与直觉一致）')

## ✏️ 练习 3：重叠区一致性监控，以及用它区分错哪个参数

实现 `overlap_monitor(pairs)`，`pairs` 是 `[(d_main, d_side), ...]`，返回 dict：

- `'n'`, `'p50'`, `'p90'` —— 相对差 $|d_1-d_2|/\min$ 的分位数
- `'by_band'` —— `{'0-20m': p90, '20-50m': p90, '50m+': p90}`（按 `d_main` 分档，无样本时 `nan`）
- `'growth'` —— 最远档 p90 / 最近档 p90（近端为 0 时取 `inf`；两端都为 0 时取 `1.0`）
- `'pattern'` —— `'ok'` / `'angular'` / `'scale'`

**判据来自几何，不是拍的**：

| 错的参数 | 读数 | 相对差随距离 |
|---|---|---|
| 相机高度（或 $f$） | $d_{\text{read}} = d\cdot h'/h$ | **恒定**（与 $d$ 无关）→ `'scale'` |
| 俯仰角 | $H/\tan(\theta+\delta)$ | **随距离增长** → `'angular'` |

所以 `pattern` 看的是 `growth` 而不是某一档的绝对值：
`p90 < 0.05` 判 `'ok'`；否则 `growth > 2` 判 `'angular'`，反之判 `'scale'`。

In [ ]:
def overlap_monitor(pairs, thresh=0.05):
    """返回 dict(n, p50, p90, by_band, growth, pattern)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
def make_pairs(pitch_err=0.0, h_side=H_CAM, dists=None):
    dists = np.linspace(8., 70., 60) if dists is None else dists
    out = []
    for d in dists:
        Pw = np.array([d, 0.5, 0.])
        a = read_range(Pw, CAM_MAIN, 0.0)
        uv = project(Pw, cam_t=CAM_SIDE)
        g = ground_from_pixel(uv, pitch=pitch_err,
                              cam_t=np.array([0., 1.0, h_side]))
        out.append((a, None if g is None else float(g[0])))
    return [(a, b) for a, b in out if b is not None]

P_OK   = make_pairs(0.0)
P_TILT = make_pairs(0.5)                       # 角度误差
P_HIGH = make_pairs(0.0, h_side=H_CAM - 0.10)  # 高度误差

for tag, pr in [('外参正确', P_OK), ('pitch 错 0.5°', P_TILT),
                ('侧相机高度错 10cm', P_HIGH)]:
    r = overlap_monitor(pr)
    assert set(r) == {'n', 'p50', 'p90', 'by_band', 'growth', 'pattern'}
    bands = ' '.join(f'{k}={100*v:5.1f}%' if v == v else f'{k}=  nan'
                     for k, v in r['by_band'].items())
    print(f'{tag:20s} p90={100*r["p90"]:5.1f}%  {bands}  '
          f'growth={r["growth"]:5.2f}  -> {r["pattern"]}')

assert overlap_monitor(P_OK)['pattern'] == 'ok'

# ★ 角度误差：相对差随距离增长
r_t = overlap_monitor(P_TILT)
assert r_t['pattern'] == 'angular', r_t
assert r_t['growth'] > 2.0, f"growth 应 > 2，实测 {r_t['growth']:.2f}"

# ★ 高度误差：相对差**精确恒定**（d_read = d·h'/h 与 d 无关）
r_h = overlap_monitor(P_HIGH)
assert r_h['pattern'] == 'scale', r_h
assert abs(r_h['growth'] - 1.0) < 1e-6, \
    f"高度误差的 growth 必须精确等于 1，实测 {r_h['growth']:.6f}"
vals = [v for v in r_h['by_band'].values() if v == v]
assert max(vals) - min(vals) < 1e-9, '各档的相对差应当逐位相同'
print(f'\n高度错 10cm：各档相对差都是 {100*vals[0]:.2f}%'
      f'（理论 {100*0.10/(H_CAM-0.10):.2f}%）—— **与距离无关**')
assert abs(vals[0] - 0.10 / (H_CAM - 0.10)) < 1e-9

print('\n✅ 练习 3 通过：**growth 把「错哪个参数」缩小到两类**')
print('   角度误差远端更糟（growth>2）· 高度/焦距误差处处相同（growth=1）')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def overlap_monitor(pairs, thresh=0.05):
    pairs = [(float(a), float(b)) for a, b in pairs]
    rel = np.array([abs(a - b) / min(a, b) for a, b in pairs])
    dm = np.array([a for a, _ in pairs])
    bands = {'0-20m': (dm < 20), '20-50m': (dm >= 20) & (dm < 50),
             '50m+': (dm >= 50)}
    by = {k: (float(np.percentile(rel[msk], 90)) if msk.sum() else float('nan'))
          for k, msk in bands.items()}
    present = [v for v in by.values() if v == v]
    near, far = present[0], present[-1]
    if near == 0 and far == 0:
        growth = 1.0
    elif near == 0:
        growth = float('inf')
    else:
        growth = far / near
    p90 = float(np.percentile(rel, 90))
    if p90 < thresh:
        pattern = 'ok'
    else:
        pattern = 'angular' if growth > 2.0 else 'scale'
    return {'n': len(pairs), 'p50': float(np.percentile(rel, 50)),
            'p90': p90, 'by_band': by, 'growth': growth, 'pattern': pattern}

assert overlap_monitor(P_OK)['pattern'] == 'ok'
assert overlap_monitor(P_TILT)['pattern'] == 'angular'
assert overlap_monitor(P_HIGH)['pattern'] == 'scale'
assert abs(overlap_monitor(P_HIGH)['growth'] - 1.0) < 1e-6
print('✅ 参考答案 3 通过')
print('   ① 报**分位数**而不是均值（个别错误关联会污染均值）；')
print('   ② growth 是免费的诊断 —— 它的依据是 d_read = d·h′/h 与距离无关，')
print('      而俯仰误差的 H/tan(θ+δ) 随距离非线性放大。')

## ✏️ 练习 4：BEV 验收器

实现 `bev_audit(grid, d_max_care, slope_worst, overlap_pairs)`，
返回 `(是否通过, {检查项: (通过?, 实测值)})`：

| 检查项 | 阈值 |
|---|---|
| `rows_per_cell_at_dmax` | `>= 0.5`（最远关心距离处每格的图像行数） |
| `ipm_range_limit_ok` | `d_max_care <= ipm_max_range(slope_worst, 0.5)` |
| `overlap_p90` | `< 0.05` |
| `overlap_pattern` | `== 'ok'` |

**用本课的默认配置跑一次——它会不通过。**

In [ ]:
def bev_audit(grid, d_max_care, slope_worst, overlap_pairs):
    """返回 (bool, {检查项: (通过?, 实测值)})。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
def show(tag, res):
    ok, det = res
    print(f'{tag} -> ' + ('通过' if ok else '**不通过**'))
    for k, (p, v) in det.items():
        vs = f'{v:.4f}' if isinstance(v, float) else str(v)
        print(f'   {"OK  " if p else "FAIL"} {k:24s} = {vs}')
    print()

# ① 本课默认：0.1m 均匀网格 + 关心 80m + 最坏 2% 坡
r_default = bev_audit(GRID, 80., 0.02, P_OK)
show('默认配置（0.1m 均匀网格 / 关心 80m / 最坏 2% 坡）', r_default)
assert r_default[0] is False, '默认配置不该通过'
assert r_default[1]['rows_per_cell_at_dmax'][0] is False, '80m 处欠采样'
assert r_default[1]['ipm_range_limit_ok'][0] is False, '80m 超出坡度允许的上限'

# ② 收缩到 IPM 真正能工作的范围
r_fixed = bev_audit(BEVGrid((0., 25.), (-10., 10.), 0.2), 25., 0.02, P_OK)
show('收缩后（0.2m 网格 / 只关心 25m / 同样 2% 坡）', r_fixed)
assert r_fixed[0] is True, r_fixed[1]

# ③ 外参漂移时被抓住
r_tilt = bev_audit(BEVGrid((0., 25.), (-10., 10.), 0.2), 25., 0.02, P_TILT)
assert r_tilt[0] is False
assert r_tilt[1]['overlap_pattern'][0] is False
print('外参漂移 0.5° -> overlap_pattern =',
      r_tilt[1]['overlap_pattern'][1], '（被抓住）')
print()
print('✅ 练习 4 通过。而最值得记的是①：'
      '**本课默认的 BEV 配置通不过自己的验收**——')
print('   0.1m 均匀网格 + 关心 80m，在采样率和坡度两项上同时不合格。')
print('   而修法不是调参数，是**缩小 IPM 的适用范围**，远处交给别的补法（模块 03）。')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def bev_audit(grid, d_max_care, slope_worst, overlap_pairs):
    # 最远关心距离处，一格对应多少图像行
    rows = abs(v_of(d_max_care) - v_of(d_max_care + grid.res))
    limit = ipm_max_range(slope_worst, 0.5)
    mon = overlap_monitor(overlap_pairs)
    checks = {
        'rows_per_cell_at_dmax': (rows >= 0.5, float(rows)),
        'ipm_range_limit_ok':    (d_max_care <= limit, float(limit)),
        'overlap_p90':           (mon['p90'] < 0.05, mon['p90']),
        'overlap_pattern':       (mon['pattern'] == 'ok', mon['pattern']),
    }
    return all(p for p, _ in checks.values()), checks

assert bev_audit(GRID, 80., 0.02, P_OK)[0] is False
assert bev_audit(BEVGrid((0., 25.), (-10., 10.), 0.2), 25., 0.02, P_OK)[0] is True
print('✅ 参考答案 4 通过')
print('   前两项只依赖配置（不需要跑数据），所以它们可以在设计评审时就算出来；')
print('   后两项需要行驶数据，属于线上监控。')

## 🧪 真实工程胶囊

```python
# ── 1) OpenCV 的 IPM ──
H_ipm = cv2.getPerspectiveTransform(src_quad, dst_quad)   # 四对点
bev   = cv2.warpPerspective(img, H_ipm, (bev_w, bev_h),
                            flags=cv2.INTER_LINEAR)        # ← 内部就是反向映射
#   ⚠️ warpPerspective 默认用 H 的**逆**去源图取值（WARP_INVERSE_MAP 控制方向）。
#      自己写循环时最容易搞反 —— 而搞反的症状就是第 2b 节那张 96.4% 空洞的图。

# ── 2) 地平线以上必须标成「无观测」，不是采样到边缘 ──
bev = cv2.warpPerspective(img, H_ipm, size, borderMode=cv2.BORDER_CONSTANT,
                          borderValue=0)
valid_mask = cv2.warpPerspective(np.ones_like(img[..., 0]), H_ipm, size,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=0)
#   ↑ 下游必须拿到 valid_mask，否则无法区分「那里是空的」与「那里没被观测」

# ── 3) IPM 函数拿类别，并拒绝不贴地的类别（第 3 节）──
GROUND_CLASSES = {'lane_marking', 'road_arrow', 'drivable_edge'}
def ipm_range(det, calib):
    if det.cls not in GROUND_CLASSES:
        raise NotGroundError(f'{det.cls} 不贴地，请用 known_size 通路')
    ...
#   ↑ 抛异常而不是返回一个数 —— 因为返回值会被静静地用下去

# ── 4) 坡度：在线估计，或写死一个作用距离上限（第 4 节）──
slope = estimate_slope_from_lane_convergence(lanes)   # 或 IMU 积分
d_limit = 0.5 * CAM_H / (WORST_SLOPE * 1.5)           # 练习 2 的公式
assert det.range_m <= d_limit, 'IPM 超出坡度允许的作用距离'

# ── 5) 重叠区一致性打点（第 6 节，唯一的线上几何指标）──
metrics.histogram('geom/overlap_rel_diff', rel, tags={'band': band})
#   报 p50/p90 + 分档；**设报警不设阻断**（它会随载重、温度正常波动）
```

> **落地顺序建议**：先加 `valid_mask`（不加它，下游的一切远处判断都在读插值），
> 再把 IPM 函数改成拿类别并抛异常，最后接重叠区一致性打点。
> <em>而「非均匀网格」放最后——它是优化，前三条是防错。</em>